<a href="https://colab.research.google.com/github/AliRaddman/digikala-ai-assistant/blob/main/notebooks/ali/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/DigikalaProject'
for d in ['raw', 'processed', 'indexes', 'models', 'eval']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print(os.listdir(BASE))

Mounted at /content/drive
['raw', 'processed', 'indexes', 'models', 'eval']


In [ ]:
!pip install -q -U "huggingface_hub[cli]"

!hf download RadeAI/Digikala_comments_products \
  --repo-type dataset \
  --revision 89c3133b169c8d3793db8834f56f32fee33d9db0 \
  --include "*.csv" \
  --local-dir /content/raw

!ls -lh /content/raw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 12.0 MB/s eta 0:00:00
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.54G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  17% 259M/1.54G [00:02<00:12, 103MB/s, 41.5MB/s  ]

Fetching 2 files:  50% 1/2 [00:02<00:02,  2.86s/it]
Reconstructing (incomplete total...):  78% 1.20G/1.54G [00:08<00:01, 199MB/s,  150MB/s  ]
Reconstructing (incomplete total...): 100% 1.54G/1.54G [00:10<00:00, 159MB/s,  199MB/s  ]

Fetching 2 files: 100% 2/2 [00:10<00:00,  5.41s/it]
Download complete: 100% 554M/554M [00:10<00:00, 85.4MB/s, 85.4MB/s  ]
Reconstruction complete: 100% 1.54G/1.54G [00:10<00:00, 159MB/s,  159MB/s  ]             ✓ Downloaded
  path: /content/raw
Download complete: 100% 

In [ ]:
import pandas as pd

prod = pd.read_csv('/content/raw/digikala-products.csv', nrows=5)
print("PRODUCTS")
print(prod.columns.tolist())
print(prod.head(3).to_string())

print("\n" + "="*80 + "\n")

com = pd.read_csv('/content/raw/digikala-comments.csv', nrows=5)
print("COMMENTS")
print(com.columns.tolist())
print(com.head(3).to_string())

PRODUCTS
['id', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']
        id                         title_fa  Rate  Rate_cnt Category1  Category2   Brand   Price         Seller  Is_Fake  min_price_last_month sub_category
0  7096438     آبسلانگ مدل s5 بسته 250 عددی    90         4   آبسلانگ        NaN  متفرقه  634800  سلامت ساز راد    False                     0       beauty
1  2845119    آبسلانگ مدل M-1 بسته 400 عددی    84       217   آبسلانگ        NaN  متفرقه  818800      مهر افزون    False                     0       beauty
2  6117745  آبسلانگ مدل m50 مجموعه 500 عددی    88        14   آبسلانگ        NaN  متفرقه  920000         یانگوم    False                     0       beauty


COMMENTS
['id', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate']
         id

In [ ]:
import pandas as pd

prod = pd.read_csv('/content/raw/digikala-products.csv', low_memory=False)
print("PRODUCTS:", prod.shape)
print("\nمقادیر گمشده:")
print(prod.isna().sum())
print("\nRate:", prod['Rate'].describe())
print("\nPrice صفر یا منفی:", (prod['Price'] <= 0).sum())
print("min_price_last_month صفر:", (prod['min_price_last_month'] == 0).sum())
print("\nid تکراری:", prod['id'].duplicated().sum())
print("\nsub_category:")
print(prod['sub_category'].value_counts().head(15))

PRODUCTS: (1283496, 12)

مقادیر گمشده:
id                           0
title_fa                     0
Rate                         0
Rate_cnt                     0
Category1                    0
Category2               210108
Brand                        0
Price                        0
Seller                     223
Is_Fake                      0
min_price_last_month         0
sub_category                 0
dtype: int64

Rate: count    1.283496e+06
mean     3.013031e+01
std      3.991098e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      8.000000e+01
max      1.000000e+02
Name: Rate, dtype: float64

Price صفر یا منفی: 223
min_price_last_month صفر: 1208195

id تکراری: 335144

sub_category:
sub_category
clothe                     587946
book & stationary & art    343597
beauty                     168648
toys and kids              118635
rural goods                 40893
travel                      23777
Name: count, dtype: int64


In [ ]:
dups = prod[prod['id'].duplicated(keep=False)].sort_values('id')
print("نمونه از یک id تکراری:")
sample_id = dups['id'].iloc[0]
print(prod[prod['id'] == sample_id].to_string())

print("\n" + "="*80)
print("\nآیا ردیف‌ها کاملاً یکسانند؟")
print("تکراری کامل (همه ستون‌ها):", prod.duplicated().sum())
print("تکراری id:", prod['id'].duplicated().sum())
print("تکراری (id, Seller):", prod.duplicated(subset=['id','Seller']).sum())
print("تکراری (id, Seller, Price):", prod.duplicated(subset=['id','Seller','Price']).sum())

نمونه از یک id تکراری:
           id                                           title_fa  Rate  Rate_cnt       Category1    Category2  Brand     Price     Seller  Is_Fake  min_price_last_month sub_category
344968  12298  ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR    64        21  اکسسوری مردانه  ساعت مردانه  کاسیو  57312000  دیجی‌کالا    False              57133000       clothe
350656  12298  ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR    64        21  اکسسوری مردانه  ساعت مردانه  کاسیو  70832000  دیجی‌کالا    False              57133000       clothe


آیا ردیف‌ها کاملاً یکسانند؟
تکراری کامل (همه ستون‌ها): 323129
تکراری id: 335144
تکراری (id, Seller): 331099
تکراری (id, Seller, Price): 326478


In [ ]:
prod_clean = prod.drop_duplicates()
print("بعد از حذف کپی کامل:", prod_clean.shape)

prod_clean = (prod_clean
              .sort_values('Price')
              .drop_duplicates(subset='id', keep='first')
              .reset_index(drop=True))
print("بعد از یکتاسازی id:", prod_clean.shape)

print("\nمحصولات با امتیاز واقعی (Rate_cnt > 0):", (prod_clean['Rate_cnt'] > 0).sum())
print("درصد:", round((prod_clean['Rate_cnt'] > 0).mean()*100, 1), "%")

print("\nقیمت (ریال):")
print(prod_clean.loc[prod_clean['Price'] > 0, 'Price'].describe())

print("\nIs_Fake:")
print(prod_clean['Is_Fake'].value_counts())

بعد از حذف کپی کامل: (960367, 12)
بعد از یکتاسازی id: (948352, 12)

محصولات با امتیاز واقعی (Rate_cnt > 0): 360933
درصد: 38.1 %

قیمت (ریال):
count    9.481540e+05
mean     9.143174e+06
std      5.187151e+07
min      2.000000e+04
25%      7.400000e+05
50%      1.700000e+06
75%      4.580000e+06
max      8.499990e+09
Name: Price, dtype: float64

Is_Fake:
Is_Fake
False    900690
True      47662
Name: count, dtype: int64


In [ ]:
import pandas as pd

cols = ['id','rate','recommendation_status','is_buyer','product_id','likes','dislikes']
n = 0
rec = pd.Series(dtype=int)
rate = pd.Series(dtype=int)

for ch in pd.read_csv('/content/raw/digikala-comments.csv', usecols=cols,
                      chunksize=1_000_000, low_memory=False):
    n += len(ch)
    rec = rec.add(ch['recommendation_status'].value_counts(dropna=False), fill_value=0)
    rate = rate.add(ch['rate'].value_counts(dropna=False), fill_value=0)
    print(f"  {n:,}")

print("\nکل نظرات:", f"{n:,}")
print("\nrecommendation_status:")
print(rec.sort_values(ascending=False))
print("\nنسبت:")
print((rec / rec.sum() * 100).round(2).sort_values(ascending=False))
print("\nrate:")
print(rate.sort_index())

  1,000,000
  2,000,000
  3,000,000
  4,000,000
  5,000,000
  6,000,000
  6,156,289

کل نظرات: 6,156,289

recommendation_status:
recommended        4162839.0
NaN                 894426.0
no_idea             594121.0
not_recommended     504903.0
dtype: float64

نسبت:
recommended        67.62
NaN                14.53
no_idea             9.65
not_recommended     8.20
dtype: float64

rate:
rate
0.00        529098.0
0.05            97.0
0.10            66.0
0.15           181.0
0.20           661.0
             ...    
4.85          1476.0
4.88             2.0
4.90           426.0
5.00       2398801.0
2500.00          1.0
Length: 178, dtype: float64


In [ ]:
import pandas as pd

n_empty_body = 0
n_short = 0
n_total = 0
lens = []

for ch in pd.read_csv('/content/raw/digikala-comments.csv',
                      usecols=['id','title','body','product_id','recommendation_status'],
                      chunksize=1_000_000, low_memory=False):
    n_total += len(ch)
    b = ch['body'].fillna('').astype(str).str.strip()
    n_empty_body += (b == '').sum()
    n_short += ((b.str.len() > 0) & (b.str.len() < 10)).sum()
    lens.append(b.str.len().describe())
    print(f"  {n_total:,}")

print("\nbody خالی:", f"{n_empty_body:,}", f"({n_empty_body/n_total*100:.1f}%)")
print("body کوتاه‌تر از ۱۰ کاراکتر:", f"{n_short:,}")
print("\nطول body (نمونه از آخرین chunk):")
print(lens[-1])

  1,000,000
  2,000,000
  3,000,000
  4,000,000
  5,000,000
  6,000,000
  6,156,289

body خالی: 637 (0.0%)
body کوتاه‌تر از ۱۰ کاراکتر: 709,736

طول body (نمونه از آخرین chunk):
count    156289.000000
mean         61.010941
std          84.161223
min           0.000000
25%          18.000000
50%          36.000000
75%          73.000000
max        4000.000000
Name: body, dtype: float64


In [ ]:
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq, os

os.makedirs('/content/processed', exist_ok=True)

def csv_to_parquet(src, dst, chunksize=500_000):
    writer, total = None, 0
    for ch in pd.read_csv(src, chunksize=chunksize, low_memory=False):
        ch = ch.astype({c: 'string' for c in ch.select_dtypes('object').columns})
        t = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(dst, t.schema, compression='zstd')
        writer.write_table(t)
        total += len(ch)
        print(f"  {total:,}")
    writer.close()
    return total

for name in ['products', 'comments']:
    src = f'/content/raw/digikala-{name}.csv'
    dst = f'/content/processed/{name}_raw.parquet'
    print(name)
    n = csv_to_parquet(src, dst)
    print(f"  {n:,} ردیف | {os.path.getsize(dst)/1e6:.0f} MB\n")

products
  500,000
  1,000,000
  1,283,496
  1,283,496 ردیف | 27 MB

comments
  500,000
  1,000,000
  1,500,000
  2,000,000
  2,500,000
  3,000,000
  3,500,000
  4,000,000
  4,500,000
  5,000,000
  5,500,000
  6,000,000
  6,156,289
  6,156,289 ردیف | 270 MB



In [ ]:
import shutil, os

DST = '/content/drive/MyDrive/DigikalaProject/raw'
for f in ['products_raw.parquet', 'comments_raw.parquet']:
    shutil.copy(f'/content/processed/{f}', f'{DST}/{f}')
    print(f, "→ ok")

!ls -lh {DST}

products_raw.parquet → ok
comments_raw.parquet → ok
total 284M
-rw------- 1 root root 258M Aug 21 22:52 comments_raw.parquet
-rw------- 1 root root  27M Aug 21 22:52 products_raw.parquet
